# Task 3 — strict ternary ResNet18 QAT (B2, no KD)

This notebook is the human-readable companion to the production Task 3 modules. It never reimplements QAT and never opens the test set.

## 1. Environment
The controlled environment is specified by `pyproject.toml` and `uv.lock`.

In [ ]:
import sys
from pathlib import Path
REPO_ROOT = Path.cwd().resolve().parent
sys.path.insert(0, str(REPO_ROOT))
!cd .. && uv run python -c "import torch; print(torch.__version__, torch.cuda.get_device_name(0))"

## 2. Repository and configuration audit

In [ ]:
from pathlib import Path
print(Path('../configs/ternary/resnet18_ternary_noKD.yaml').read_text())
print(Path('../experiments/task3/EXPERIMENT_MATRIX.md').read_text())

## 3. Task 2 baseline loading
Only the frozen seed-43 validation-best Task 2 checkpoint is used for the canonical warm-start.

In [ ]:
import torch
t2 = torch.load('../results/best_models/task2_b1_fp32_resnet18_best.pth', map_location='cpu', weights_only=False)
{k: t2[k] for k in ('arch', 'seed', 'epoch', 'best_validation_accuracy', 'selection_metric')}

## 4. FP32 ResNet18 inspection

In [ ]:
from src.models.resnet_cifar import resnet18_cifar, count_parameters
fp32 = resnet18_cifar(); fp32.load_state_dict(t2['model_state_dict']); count_parameters(fp32)

## 5. Ternary conversion
The production conversion replaces all 20 convolutions and the final FC, including `conv1`.

In [ ]:
from src.quant.ternary import QuantConfig, convert_to_ternary, count_ternary_layers
qat = convert_to_ternary(fp32, QuantConfig()); count_ternary_layers(qat)

## 6. Latent versus deployed visualization

In [ ]:
import matplotlib.pyplot as plt
layer = qat.conv1
plt.hist(layer.weight.detach().flatten().numpy(), bins=80, alpha=.6, label='latent FP32')
plt.hist(layer.deployed_weight().detach().flatten().numpy(), bins=3, alpha=.6, label='deployed ternary')
plt.legend(); plt.show()

## 7. Ternary verification

In [ ]:
from src.evaluation.verify_ternary import verify_model
verification = verify_model(qat, verbose=False); verification['all_ternary'], verification['all_weight_layers_ternary']

## 8. One-batch forward/backward test

In [ ]:
!cd .. && uv run scripts/run_task3_smoke_test.py

## 9. QAT smoke test
Uses production trainer, training/validation only.

In [ ]:
# !cd .. && uv run scripts/train_student_ternary.py --run-name task3_trainer_smoke --seeds 42 --smoke

## 10. Baseline configuration
B2-default: per-channel symmetric TWN, detached derived alpha, clipped STE, warm-start, all conv/FC ternary, no KD.

## 11. Candidate improvement experiments
The predefined screens vary exactly one factor at a time; see `EXPERIMENT_MATRIX.md`.

## 12. Short-training comparison
Run only after smoke passes; results are read from `results/task3_screening_summary.json`.

In [ ]:
import json
screen = Path('../results/task3_screening_summary.json')
json.loads(screen.read_text()) if screen.exists() else 'Screening not run yet.'

## 13. Final Task 3 3-seed training
This command is intentionally commented until short-screening records are complete.

In [ ]:
# !cd .. && uv run scripts/train_student_ternary.py --config configs/ternary/resnet18_ternary_noKD.yaml

## 14. Per-seed results
## 15. Mean ± standard deviation
## 16. Quantization statistics
## 17. Layer-wise sparsity
## 18. Alpha and threshold distributions
## 19. Quantization error
## 20. Accuracy gap versus Task 2
## 21. Theoretical compression
## 22. Final conclusions

The cells below only read saved, validation-only production artifacts.

In [ ]:
from pprint import pprint
summary_path = Path('../results/task3_b2_default_summary.json')
if summary_path.exists():
    summary = json.loads(summary_path.read_text())
    pprint(summary)
else:
    print('Final B2-default training has not run yet; no test accuracy is available or requested.')